## Tests — OpenAI API Connection (Categorize + Coach)

Live connection checks for every project path that calls OpenAI:

1. **Categorize gateway** — same key / base URL / model as categorize
2. **Coach** — `model.coach_core.make_openai_chat_call`

Each section sends one tiny `"hello"` chat (not a full categorize batch). Requires project-root `.env`:
`LLM_API_KEY` (or `OPENAI_API_KEY`), `LLM_MODEL`, and `OPENAI_BASE_URL` / `LLM_API_BASE` if using a gateway.


## Imports + env

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir() and (candidate / "artifacts").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/ and artifacts/")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env", override=True)

API_KEY = (os.environ.get("LLM_API_KEY") or os.environ.get("OPENAI_API_KEY") or "").strip()
LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-4o-mini")
BASE_URL = os.environ.get("OPENAI_BASE_URL") or os.environ.get("LLM_API_BASE")

assert API_KEY, (
    f"Missing LLM_API_KEY / OPENAI_API_KEY after loading {(ROOT / '.env')}"
)

from model.coach_core import make_openai_chat_call

print("Project root:", ROOT)
print("Model:", LLM_MODEL)
print("API key present:", True, f"(len={len(API_KEY)})")
print("Base URL:", BASE_URL or "(default OpenAI)")


Project root: /Users/nicholasp/Personal Coding/JHU/personal finance
Model: gpt-4o-mini
API key present: True (len=67)
Base URL: https://aibe.mygreatlearning.com/openai/v1


## 1) Categorize gateway connection

One tiny `"hello"` chat using the same API key / base URL / model as categorize.


In [2]:
from openai import OpenAI

client_kwargs = {"api_key": API_KEY}
if BASE_URL:
    client_kwargs["base_url"] = BASE_URL
client = OpenAI(**client_kwargs)

try:
    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": "hello"}],
            max_tokens=16,
            temperature=0,
        )
    except TypeError:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": "hello"}],
            max_completion_tokens=16,
            temperature=0,
        )
except Exception as exc:  # noqa: BLE001
    msg = str(exc)
    if "429" in msg or "quota" in msg.lower():
        raise RuntimeError(
            "Categorize gateway hit quota/rate limit (HTTP 429). "
            "Auth/base URL work, but this gateway has no remaining quota. "
            f"Details: {exc}"
        ) from exc
    raise

reply = (resp.choices[0].message.content or "").strip()
assert reply, repr(resp)

print("Categorize gateway connection: OK")
print("Reply:", repr(reply))


RuntimeError: Categorize gateway hit quota/rate limit (HTTP 429). Auth/base URL work, but this gateway has no remaining quota. Details: Error code: 429 - {'reason': {'error': 'You exceeded your current quota!!'}}

## 2) Coach connection (`make_openai_chat_call`)

One tiny `"hello"` chat through the coach client factory.


In [ ]:
try:
    coach_llm = make_openai_chat_call(model=LLM_MODEL)
    reply = coach_llm([{"role": "user", "content": "hello"}])
except Exception as exc:  # noqa: BLE001
    msg = str(exc)
    if "429" in msg or "quota" in msg.lower():
        raise RuntimeError(
            "Coach connection hit quota/rate limit (HTTP 429). "
            "Auth/base URL work, but this gateway has no remaining quota. "
            f"Details: {exc}"
        ) from exc
    raise

assert isinstance(reply, str) and reply.strip(), repr(reply)

print("Coach connection: OK")
print("Reply:", repr(reply))
print("LLM connection tests: PASS")
